# RIFT-LoRA Week 8: Qwen2.5-3B classification matrix

This notebook clones the published RIFT-LoRA repository and runs the reproducible Week 8 matrix on Kaggle. It covers SST-2, QNLI, MNLI matched, and MNLI mismatched under IID homogeneous, IID heterogeneous-rank, and non-IID high-staleness regimes.

The full matrix is 4 tasks x 3 regimes x 8 methods x 6 seeds = 576 runs. Use `focused` first on T4x2; switch to `full` only when the smoke result and GPU memory are healthy. The launcher runs at most one QLoRA job per visible T4.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time

REPO_URL = 'https://github.com/TrgPhan/RIFTLoRA.git'
REPO_REF = 'b2e2ae6'  # Published Week 8 matrix commit.
RUN_MODE = 'focused'  # smoke, focused, or full
FORCE_RERUN = False
CACHE_ASSETS = True
WORK_ROOT = Path('/kaggle/working')
REPO_DIR = WORK_ROOT / 'RIFTLoRA'
assert REPO_DIR.parent == WORK_ROOT
print({'run_mode': RUN_MODE, 'repo_ref': REPO_REF, 'workdir': str(REPO_DIR)})

In [ ]:
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', 'checkout', REPO_REF], cwd=REPO_DIR, check=True)
resolved_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
print('Resolved repository commit:', resolved_commit)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[scale]'], cwd=REPO_DIR, check=True)
print('Installed package from', REPO_DIR)

In [ ]:
subprocess.run(['nvidia-smi'], check=False)
import torch
gpu_count = torch.cuda.device_count()
GPU_IDS = list(range(min(gpu_count, 2)))
if not GPU_IDS:
    raise RuntimeError('A CUDA GPU is required for the Qwen2.5-3B matrix.')
print({'cuda': torch.version.cuda, 'gpu_count': gpu_count, 'worker_gpu_ids': GPU_IDS})

In [ ]:
# Cache the model and GLUE datasets up front so later failures are easier to diagnose.
if CACHE_ASSETS:
    from huggingface_hub import snapshot_download
    from datasets import load_dataset
    snapshot_download('Qwen/Qwen2.5-3B-Instruct')
    load_dataset('glue', 'sst2')
    load_dataset('glue', 'qnli')
    load_dataset('glue', 'mnli')
    print('Model and SST-2/QNLI/MNLI assets are cached.')
else:
    print('Asset pre-cache disabled; the runner will download on demand.')

In [ ]:
# Catch packaging/import regressions before spending GPU time.
subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=REPO_DIR, check=True)
subprocess.run([sys.executable, '-m', 'py_compile', 'scripts/run_kaggle_3b.py', 'scripts/run_week8_classification_matrix.py', 'scripts/analyze_kaggle_3b_rift_competitors.py'], cwd=REPO_DIR, check=True)
print('Repository tests and runner compilation passed.')

## Select the experiment budget

`smoke` runs three methods on the hardest SST-2 slice with one seed. `focused` runs every method on the hard slice for all four tasks and three seeds. `full` runs all 576 manifest combinations and is the only mode that can produce a complete Week 8 GO/NO-GO decision.

In [ ]:
MATRIX = REPO_DIR / 'configs' / 'week8_rift_classification_matrix.json'
matrix = json.loads(MATRIX.read_text())
task_names = [task['name'] for task in matrix['tasks']]
regime_names = [regime['name'] for regime in matrix['regimes']]
method_names = list(matrix['methods'])
seed_names = matrix['seeds']
if RUN_MODE == 'smoke':
    selected_tasks = ['sst2']
    selected_regimes = ['noniid_high_staleness']
    selected_methods = ['raw', 'freshness', 'rift']
    selected_seeds = [seed_names[0]]
elif RUN_MODE == 'focused':
    selected_tasks = task_names
    selected_regimes = ['noniid_high_staleness']
    selected_methods = method_names
    selected_seeds = seed_names[:3]
elif RUN_MODE == 'full':
    selected_tasks = task_names
    selected_regimes = regime_names
    selected_methods = method_names
    selected_seeds = seed_names
else:
    raise ValueError('RUN_MODE must be smoke, focused, or full')
jobs = [(task, regime, method, seed) for task in selected_tasks for regime in selected_regimes for method in selected_methods for seed in selected_seeds]
print({'tasks': selected_tasks, 'regimes': selected_regimes, 'methods': selected_methods, 'seeds': selected_seeds, 'job_count': len(jobs), 'gpu_workers': len(GPU_IDS)})

In [ ]:
MATRIX_RUNNER = REPO_DIR / 'scripts' / 'run_week8_classification_matrix.py'
sample_job = jobs[0]
dry_run_cmd = [sys.executable, str(MATRIX_RUNNER), '--matrix', str(MATRIX), '--task', sample_job[0], '--regime', sample_job[1], '--method', sample_job[2], '--seed', str(sample_job[3]), '--dry-run']
subprocess.run(dry_run_cmd, cwd=REPO_DIR, check=True)

In [ ]:
# Run two independent one-GPU workers when T4x2 is available. Existing result.json files are reused.
LOG_DIR = REPO_DIR / 'outputs' / 'week8_kaggle_logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)
processes = []
failures = []

def start_job(job, gpu_id):
    task, regime, method, seed = job
    command = [sys.executable, str(MATRIX_RUNNER), '--matrix', str(MATRIX), '--task', task, '--regime', regime, '--method', method, '--seed', str(seed)]
    if FORCE_RERUN:
        command.append('--force')
    log_path = LOG_DIR / f'{task}_{regime}_{method}_seed{seed}_gpu{gpu_id}.log'
    handle = log_path.open('w')
    env = os.environ.copy()
    env['CUDA_VISIBLE_DEVICES'] = str(gpu_id)
    process = subprocess.Popen(command, cwd=REPO_DIR, env=env, stdout=handle, stderr=subprocess.STDOUT, text=True)
    return process, handle, job, log_path

for offset in range(0, len(jobs), len(GPU_IDS)):
    wave = jobs[offset:offset + len(GPU_IDS)]
    processes = [start_job(job, GPU_IDS[index]) for index, job in enumerate(wave)]
    while any(process.poll() is None for process, _, _, _ in processes):
        time.sleep(2)
    for process, handle, job, log_path in processes:
        handle.close()
        code = process.returncode
        print({'job': job, 'return_code': code, 'log': str(log_path)})
        if code != 0:
            failures.append((job, str(log_path)))
if failures:
    raise RuntimeError(f'{len(failures)} jobs failed; inspect the listed logs.')
print('Completed jobs:', len(jobs))

In [ ]:
ANALYZER = REPO_DIR / 'scripts' / 'analyze_kaggle_3b_rift_competitors.py'
RESULT_DIR = REPO_DIR / 'outputs' / 'week8_classification_matrix'
ANALYSIS_DIR = REPO_DIR / 'outputs' / 'week8_analysis'
analysis_command = [sys.executable, str(ANALYZER), '--input-dir', str(RESULT_DIR), '--output-dir', str(ANALYSIS_DIR), '--matrix', str(MATRIX)]
analysis_process = subprocess.run(analysis_command, cwd=REPO_DIR, text=True, capture_output=True)
print(analysis_process.stdout)
print(analysis_process.stderr)
if analysis_process.returncode != 0:
    raise RuntimeError('The analysis command failed.')
report_path = ANALYSIS_DIR / 'week8_verdict.md'
if report_path.exists():
    print(report_path.read_text())
else:
    print('No verdict report was produced; inspect analyzer output.')

In [ ]:
# Package outputs for download from the Kaggle notebook UI.
archive_base = WORK_ROOT / f'riftlora-week8-{RUN_MODE}-results'
archive_path = Path(shutil.make_archive(str(archive_base), 'zip', root_dir=REPO_DIR / 'outputs'))
metadata = {'repo_url': REPO_URL, 'repo_commit': resolved_commit, 'run_mode': RUN_MODE, 'job_count': len(jobs), 'gpu_ids': GPU_IDS}
(WORK_ROOT / f'riftlora-week8-{RUN_MODE}-metadata.json').write_text(json.dumps(metadata, indent=2))
print('Archive:', archive_path)
print('Metadata:', metadata)

## Reading the result

The analyzer reports `GO`, `NO-GO`, or `INCONCLUSIVE`. A partial smoke/focused run is intentionally `INCONCLUSIVE` because the Week 8 acceptance gate requires all task/regime/method/seed cells, paired seeds, accuracy and NLL non-inferiority, and positive late-harm reduction on the hard slice. MNLI is three-class: use its multiclass NLL/Brier/accuracy outputs; binary candidate NLL is not interpreted for MNLI.